# 🎬 Movie Recommendation Engine
### Content-Based Filtering using TF-IDF Vectorization & Cosine Similarity
---
**Dataset:** `movies.csv` (4,803 movies)  
**Approach:** Content-Based Filtering  
**Core Technique:** TF-IDF → Cosine Similarity  
**Features Used:** `genres`, `keywords`, `tagline`, `cast`, `director`

## 📐 Concept: Cosine Similarity

Cosine similarity measures the angle between two vectors in n-dimensional space.  
It is defined as the **dot product divided by the product of the magnitudes**.

**Formula:**  
$$\text{cosine\_similarity} = \frac{\vec{A} \cdot \vec{B}}{|\vec{A}| \cdot |\vec{B}|}$$

**Example:**
- `vec1 = [1,1,0,1,1]`  
- `vec2 = [0,1,0,1,1]`
- Dot product = 0+1+0+1+1 = **3**  
- |vec1| = √(1+1+0+1+1) = √4 = **2**  
- |vec2| = √(0+1+0+1+1) = √3 ≈ **1.73**  
- Cosine Similarity = 3 / (2 × 1.73) ≈ **0.867**  

A score of **1.0** means identical direction (same movie); **0.0** means no similarity.

In [10]:
from sklearn.metrics.pairwise import cosine_similarity

# Quick demonstration of cosine similarity with the example from above
vec1 = [1, 1, 0, 1, 1]
vec2 = [0, 1, 0, 1, 1]

result = cosine_similarity([vec1, vec2])
print("Cosine Similarity Matrix:")
print(result)
print(f"\nSimilarity between vec1 and vec2: {result[0][1]:.4f}")

Cosine Similarity Matrix:
[[1.        0.8660254]
 [0.8660254 1.       ]]

Similarity between vec1 and vec2: 0.8660


## 📚 Step 1 – Import Libraries

In [11]:
# Core libraries
import numpy as np
import pandas as pd

# difflib: used to find close matches for user-entered movie names
# Handles spelling mistakes and partial matches
import difflib

# TfidfVectorizer: converts raw text into numerical TF-IDF feature vectors
# Each word gets a score based on frequency in one document vs. all documents
from sklearn.feature_extraction.text import TfidfVectorizer

# cosine_similarity: computes similarity between TF-IDF vectors
from sklearn.metrics.pairwise import cosine_similarity

import os
import warnings
warnings.filterwarnings('ignore')

print("✅ Libraries imported successfully.")

✅ Libraries imported successfully.


## 📂 Step 2 – Load Dataset

In [12]:
from google.colab import files

print("Please select the 'movies.csv' file from your local computer to upload:")
uploaded = files.upload()

for fn in uploaded.keys():
  print(f'User uploaded file "{fn}" with length {len(uploaded[fn])} bytes')


Please select the 'movies.csv' file from your local computer to upload:


Saving movies.csv to movies.csv
User uploaded file "movies.csv" with length 23432179 bytes


In [13]:
# Check current directory
display(os.getcwd())

# Update the path below if movies.csv is in a different folder
# os.chdir('C:\\Your\\Path\\To\\Project\\')

# To resolve FileNotFoundError:
# Option 1: Upload 'movies.csv' to the current Colab environment (e.g., using the file explorer on the left).
# Option 2: If 'movies.csv' is in Google Drive, mount Drive and provide the full path:
# from google.colab import drive
# drive.mount('/content/drive')
# movies_data = pd.read_csv('/content/drive/MyDrive/path/to/movies.csv')

# Load the movies dataset (4,803 movies with metadata)
movies_data = pd.read_csv('movies.csv')

print("Dataset loaded — first 5 rows:")
display(movies_data.head())

'/content'

Dataset loaded — first 5 rows:


,index,budget,genres,homepage,id,keywords,original_language,original_title,overview,popularity,...,runtime,spoken_languages,status,tagline,title,vote_average,vote_count,cast,crew,director
0,0,237000000,Action Adventure Fantasy Science Fiction,http://www.avatarmovie.com/,19995,culture clash future space war space colony so...,en,Avatar,"In the 22nd century, a paraplegic Marine is di...",150.437577,...,162.0,"[{""iso_639_1"": ""en"", ""name"": ""English""}, {""iso...",Released,Enter the World of Pandora.,Avatar,7.2,11800,Sam Worthington Zoe Saldana Sigourney Weaver S...,"[{'name': 'Stephen E. Rivkin', 'gender': 0, 'd...",James Cameron
1,1,300000000,Adventure Fantasy Action,http://disney.go.com/disneypictures/pirates/,285,ocean drug abuse exotic island east india trad...,en,Pirates of the Caribbean: At World's End,"Captain Barbossa, long believed to be dead, ha...",139.082615,...,169.0,"[{""iso_639_1"": ""en"", ""name"": ""English""}]",Released,"At the end of the world, the adventure begins.",Pirates of the Caribbean: At World's End,6.9,4500,Johnny Depp Orlando Bloom Keira Knightley Stel...,"[{'name': 'Dariusz Wolski', 'gender': 2, 'depa...",Gore Verbinski
2,2,245000000,Action Adventure Crime,http://www.sonypictures.com/movies/spectre/,206647,spy based on novel secret agent sequel mi6,en,Spectre,A cryptic message from Bond’s past sends him o...,107.376788,...,148.0,"[{""iso_639_1"": ""fr"", ""name"": ""Fran\u00e7ais""},...",Released,A Plan No One Escapes,Spectre,6.3,4466,Daniel Craig Christoph Waltz L\u00e9a Seydoux ...,"[{'name': 'Thomas Newman', 'gender': 2, 'depar...",Sam Mendes
3,3,250000000,Action Crime Drama Thriller,http://www.thedarkknightrises.com/,49026,dc comics crime fighter terrorist secret ident...,en,The Dark Knight Rises,Following the death of District Attorney Harve...,112.312950,...,165.0,"[{""iso_639_1"": ""en"", ""name"": ""English""}]",Released,The Legend Ends,The Dark Knight Rises,7.6,9106,Christian Bale Michael Caine Gary Oldman Anne ...,"[{'name': 'Hans Zimmer', 'gender': 2, 'departm...",Christopher Nolan
4,4,260000000,Action Adventure Science Fiction,http://movies.disney.com/john-carter,49529,based on novel mars medallion space travel pri...,en,John Carter,"John Carter is a war-weary, former military ca...",43.926995,...,132.0,"[{""iso_639_1"": ""en"", ""name"": ""English""}]",Released,"Lost in our world, found in another.",John Carter,6.1,2124,Taylor Kitsch Lynn Collins Samantha Morton Wil...,"[{'name': 'Andrew Stanton', 'gender': 2, 'depa...",Andrew Stanton


## 🧪 Step 3 – Exploratory Data Analysis

In [14]:
# Dataset dimensions: rows = movies, columns = metadata features
print("Dataset Shape (movies × features):")
display(movies_data.shape)

Dataset Shape (movies × features):


(4803, 24)

In [15]:
# Column names, data types, and non-null counts
print("Dataset Info:")
display(movies_data.info())

Dataset Info:
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 4803 entries, 0 to 4802
Data columns (total 24 columns):
 #   Column                Non-Null Count  Dtype  
---  ------                --------------  -----  
 0   index                 4803 non-null   int64  
 1   budget                4803 non-null   int64  
 2   genres                4775 non-null   object 
 3   homepage              1712 non-null   object 
 4   id                    4803 non-null   int64  
 5   keywords              4391 non-null   object 
 6   original_language     4803 non-null   object 
 7   original_title        4803 non-null   object 
 8   overview              4800 non-null   object 
 9   popularity            4803 non-null   float64
 10  production_companies  4803 non-null   object 
 11  production_countries  4803 non-null   object 
 12  release_date          4802 non-null   object 
 13  revenue               4803 non-null   int64  
 14  runtime               4801 non-null   float64
 15  spoken_

None

In [16]:
# Count missing values across all columns
# Several text columns (tagline, cast, keywords) commonly have nulls
print("Missing Values per Column:")
display(movies_data.isna().sum())

Missing Values per Column:


,0
index,0
budget,0
genres,28
homepage,3091
id,0
keywords,412
original_language,0
original_title,0
overview,3
popularity,0


## 🎯 Step 4 – Select Relevant Features for Recommendation
We use only the 5 content-rich text features that best describe what a movie is **about**.

In [17]:
# These 5 features capture the essence of a movie's content:
#   genres    – Action, Drama, Comedy etc.
#   keywords  – plot keywords like 'space', 'superhero'
#   tagline   – marketing phrase (e.g., "Just when you thought it was safe...")
#   cast      – lead actors
#   director  – director name (strong signal for film style)
selected_features = ['genres', 'keywords', 'tagline', 'cast', 'director']
print("Selected Features:", selected_features)

print("\nSample of selected columns:")
display(movies_data[selected_features].head())

Selected Features: ['genres', 'keywords', 'tagline', 'cast', 'director']

Sample of selected columns:


,genres,keywords,tagline,cast,director
0,Action Adventure Fantasy Science Fiction,culture clash future space war space colony so...,Enter the World of Pandora.,Sam Worthington Zoe Saldana Sigourney Weaver S...,James Cameron
1,Adventure Fantasy Action,ocean drug abuse exotic island east india trad...,"At the end of the world, the adventure begins.",Johnny Depp Orlando Bloom Keira Knightley Stel...,Gore Verbinski
2,Action Adventure Crime,spy based on novel secret agent sequel mi6,A Plan No One Escapes,Daniel Craig Christoph Waltz L\u00e9a Seydoux ...,Sam Mendes
3,Action Crime Drama Thriller,dc comics crime fighter terrorist secret ident...,The Legend Ends,Christian Bale Michael Caine Gary Oldman Anne ...,Christopher Nolan
4,Action Adventure Science Fiction,based on novel mars medallion space travel pri...,"Lost in our world, found in another.",Taylor Kitsch Lynn Collins Samantha Morton Wil...,Andrew Stanton


In [18]:
# Confirm nulls only in the selected columns
print("Missing Values in Selected Features:")
display(movies_data[selected_features].isna().sum())

Missing Values in Selected Features:


,0
genres,28
keywords,412
tagline,844
cast,43
director,30


## 🔧 Step 5 – Handle Missing Values

In [19]:
# Replace NaN with empty string '' so string concatenation works without errors
# If we left NaN, combining features would produce 'nan' as literal text
for feature in selected_features:
    movies_data[feature] = movies_data[feature].fillna('')

print("After null-fill — missing values:")
display(movies_data[selected_features].isna().sum())

After null-fill — missing values:


,0
genres,0
keywords,0
tagline,0
cast,0
director,0


## 🔗 Step 6 – Combine Features into a Single Text String
Concatenate the 5 features into one string per movie. This becomes the input document for TF-IDF.

In [20]:
# Merge all 5 features into a single text blob per movie
# Example result for Avatar: "Action Adventure...alien planet...Enter a world...Sam Worthington...James Cameron"
combined_features = (
    movies_data['genres'] + ' ' +
    movies_data['keywords'] + ' ' +
    movies_data['tagline'] + ' ' +
    movies_data['cast'] + ' ' +
    movies_data['director']
)

print("Combined feature text (first 3 movies):")
display(combined_features.head(3))

Combined feature text (first 3 movies):


,0
0,Action Adventure Fantasy Science Fiction cultu...
1,Adventure Fantasy Action ocean drug abuse exot...
2,Action Adventure Crime spy based on novel secr...


## 📊 Step 7 – TF-IDF Vectorization

**TF-IDF (Term Frequency–Inverse Document Frequency)** converts raw text to numerical vectors:

- **TF(t, d)** = count of term `t` in document `d` / total words in `d`
- **IDF(t)** = 1 + log(n / df(t)) where `n` = total documents, `df(t)` = docs containing `t`
- **TF-IDF = TF × IDF**

> Words common to all movies ("the", "a") get low IDF. Rare but meaningful words get high IDF.  
> sklearn applies L2 normalization and smoothing, so values may differ from manual calculations.

In [21]:
# TfidfVectorizer converts the combined text into a sparse matrix
# Output shape: (4803 movies, N unique words)
# Each cell = TF-IDF score of a word in that movie's combined text
vectorizer = TfidfVectorizer()
feature_vectors = vectorizer.fit_transform(combined_features)

print("TF-IDF Matrix shape (movies × unique words):")
display(feature_vectors.shape)

# Dense view of first 5 movies × first 10 words
print("\nSample TF-IDF values (first 5 rows, first 10 features):")
display(pd.DataFrame(feature_vectors.toarray()).iloc[:5, :10])

TF-IDF Matrix shape (movies × unique words):


(4803, 17318)


Sample TF-IDF values (first 5 rows, first 10 features):


,0,1,2,3,4,5,6,7,8,9
0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
1,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
2,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
3,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
4,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0


## 🔁 Step 8 – Compute Cosine Similarity Matrix
Calculate pairwise cosine similarity between all 4,803 movies.  
Result: a 4803 × 4803 matrix where `similarity[i][j]` = similarity score between movie i and movie j.

In [22]:
# Compute the full cosine similarity matrix
# This is computationally intensive — may take a few seconds
# similarity[i][j] ranges from 0 (no similarity) to 1 (identical)
similarity = cosine_similarity(feature_vectors)

print("Similarity Matrix Shape:", similarity.shape)
print("\nSample (top-left 5×5 corner):")
print(similarity[:5, :5])

Similarity Matrix Shape: (4803, 4803)

Sample (top-left 5×5 corner):
[[1.         0.07219487 0.037733   0.0125202  0.10702574]
 [0.07219487 1.         0.03281499 0.0207415  0.03305463]
 [0.037733   0.03281499 1.         0.05179498 0.05772601]
 [0.0125202  0.0207415  0.05179498 1.         0.00671708]
 [0.10702574 0.03305463 0.05772601 0.00671708 1.        ]]


## 🎥 Step 9 – Movie Recommendation Function
Given a movie name (with fuzzy matching), find the 30 most similar movies.

In [23]:
# Build a list of all movie titles from the dataset for fuzzy matching
list_of_all_titles = movies_data['title'].tolist()

print(f"Total movies in dataset: {len(list_of_all_titles)}")
print("\nSample titles:", list_of_all_titles[:10])

Total movies in dataset: 4803

Sample titles: ['Avatar', "Pirates of the Caribbean: At World's End", 'Spectre', 'The Dark Knight Rises', 'John Carter', 'Spider-Man 3', 'Tangled', 'Avengers: Age of Ultron', 'Harry Potter and the Half-Blood Prince', 'Batman v Superman: Dawn of Justice']


In [24]:
def recommend_movies(movie_name, top_n=30):
    """
    Recommend movies similar to the given movie title.

    Parameters:
    -----------
    movie_name : str
        The name of a movie (can be approximate / misspelled)
    top_n : int
        Number of similar movies to return (default: 30)

    How it works:
    -------------
    1. difflib.get_close_matches finds the nearest title in the dataset
    2. We get that movie's row index
    3. We extract its similarity scores against all other movies
    4. We sort by score (descending) and print the top N titles
    """

    # Step 1: Fuzzy match the input name to the closest title in the dataset
    # cutoff=0.6 means at least 60% similar characters
    find_close_match = difflib.get_close_matches(movie_name, list_of_all_titles)

    if not find_close_match:
        print(f"❌ No close match found for '{movie_name}'. Please try a different title.")
        return

    close_match = find_close_match[0]
    print(f"🎯 Closest match found: '{close_match}'")

    # Step 2: Retrieve the dataset index for the matched movie
    index_of_the_movie = movies_data[movies_data.title == close_match]['index'].values[0]

    # Step 3: Get the similarity scores for this movie vs. all others
    # enumerate adds the movie index alongside its score
    similarity_score = list(enumerate(similarity[index_of_the_movie]))

    # Step 4: Sort movies by similarity score in descending order
    sorted_similar_movies = sorted(similarity_score, key=lambda x: x[1], reverse=True)

    # Step 5: Print the top N similar movies (skip index 0 = the movie itself)
    print(f"\n🎬 Movies suggested for you (based on '{close_match}'):\n")
    i = 1
    for movie in sorted_similar_movies:
        movie_idx = movie[0]
        title_from_index = movies_data[movies_data.index == movie_idx]['title'].values[0]
        if i <= top_n:
            print(f"  {i:2}. {title_from_index}  (similarity: {movie[1]:.4f})")
            i += 1

print("✅ recommend_movies() function defined.")

✅ recommend_movies() function defined.


## ▶️ Step 10 – Run the Recommendation Engine
Enter any movie name below to get recommendations!

In [25]:
# --- Static Examples ---
# Run pre-defined recommendations (no input needed)

recommend_movies('Avatar')
print("\n" + "="*60 + "\n")
recommend_movies('The Dark Knight', top_n=10)

🎯 Closest match found: 'Avatar'

🎬 Movies suggested for you (based on 'Avatar'):

   1. Avatar  (similarity: 1.0000)
   2. Alien  (similarity: 0.2495)
   3. Aliens  (similarity: 0.2484)
   4. Guardians of the Galaxy  (similarity: 0.2451)
   5. Star Trek Beyond  (similarity: 0.2038)
   6. Star Trek Into Darkness  (similarity: 0.2012)
   7. Galaxy Quest  (similarity: 0.1970)
   8. Alien³  (similarity: 0.1802)
   9. Cargo  (similarity: 0.1765)
  10. Trekkies  (similarity: 0.1745)
  11. Gravity  (similarity: 0.1744)
  12. Moonraker  (similarity: 0.1683)
  13. Jason X  (similarity: 0.1650)
  14. Pocahontas  (similarity: 0.1606)
  15. Space Cowboys  (similarity: 0.1564)
  16. The Helix... Loaded  (similarity: 0.1547)
  17. Lockout  (similarity: 0.1522)
  18. Event Horizon  (similarity: 0.1503)
  19. Space Dogs  (similarity: 0.1493)
  20. Machete Kills  (similarity: 0.1466)
  21. Gettysburg  (similarity: 0.1448)
  22. Clash of the Titans  (similarity: 0.1423)
  23. Star Wars: Clone Wars: Volu

In [26]:
# --- Interactive Mode ---
# Uncomment the block below to enter a movie name dynamically

# movie_name = input('Enter your favourite movie name: ')
# recommend_movies(movie_name)

## 📋 Step 11 – Consolidated Single-Block Code
All steps in one executable cell for quick deployment or script conversion.

In [27]:
# ============================================================
# CONSOLIDATED: Full Movie Recommendation Pipeline
# ============================================================

import numpy as np, pandas as pd, difflib
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity

# Load data
movies_data = pd.read_csv('movies.csv')

# Fill nulls in selected feature columns
selected_features = ['genres', 'keywords', 'tagline', 'cast', 'director']
for feature in selected_features:
    movies_data[feature] = movies_data[feature].fillna('')

# Combine features into single text per movie
combined_features = (
    movies_data['genres'] + ' ' + movies_data['keywords'] + ' ' +
    movies_data['tagline'] + ' ' + movies_data['cast'] + ' ' + movies_data['director']
)

# Build TF-IDF matrix and cosine similarity
vectorizer = TfidfVectorizer()
feature_vectors = vectorizer.fit_transform(combined_features)
similarity = cosine_similarity(feature_vectors)
list_of_all_titles = movies_data['title'].tolist()

# --- Get user input and generate recommendations ---
movie_name = input(' Enter your favourite movie name: ')

find_close_match = difflib.get_close_matches(movie_name, list_of_all_titles)
close_match = find_close_match[0]
index_of_the_movie = movies_data[movies_data.title == close_match]['index'].values[0]
similarity_score = list(enumerate(similarity[index_of_the_movie]))
sorted_similar_movies = sorted(similarity_score, key=lambda x: x[1], reverse=True)

print(f'Movies suggested for you (based on "{close_match}"):\n')
i = 1
for movie in sorted_similar_movies:
    index = movie[0]
    title_from_index = movies_data[movies_data.index == index]['title'].values[0]
    if i < 30:
        print(i, '.', title_from_index)
        i += 1

 Enter your favourite movie name: Avatar
Movies suggested for you (based on "Avatar"):

1 . Avatar
2 . Alien
3 . Aliens
4 . Guardians of the Galaxy
5 . Star Trek Beyond
6 . Star Trek Into Darkness
7 . Galaxy Quest
8 . Alien³
9 . Cargo
10 . Trekkies
11 . Gravity
12 . Moonraker
13 . Jason X
14 . Pocahontas
15 . Space Cowboys
16 . The Helix... Loaded
17 . Lockout
18 . Event Horizon
19 . Space Dogs
20 . Machete Kills
21 . Gettysburg
22 . Clash of the Titans
23 . Star Wars: Clone Wars: Volume 1
24 . The Right Stuff
25 . Terminator Salvation
26 . The Astronaut's Wife
27 . Planet of the Apes
28 . Star Trek
29 . Wing Commander
